# Notebook 8: Auto Router Adapter Prediction

Bu notebook tek goruntu icin tam akisi calistirir: Notebook 1 router yuzeyi crop/part secer, sonra secilen adapter otomatik yuklenir ve hastalik/OOD tahmini uretilir.

Router davranisi Notebook 1'in maintained cell script'lerinden gelir. Notebook 8 sadece router sonucundan sonraki adapter tahmin adimini ekleyen ince wrapper'dir.


## Hizli Akis

- Public repo varsayilani tek yuklenen gorsel uzerinde router + adapter tahminidir; tum hucreleri sirayla calistirin.
- `M2_DEMO_MANIFEST = 'docs/demo_assets/m2_full_image_set/manifests/m2_balanced_80_run_manifest.csv'` balanced supported/safety evidence setini kosar; 48-row customer demo kaniti `docs/demo_results/m2/20260706T153334Z/` altinda korunur.
- `M2_FORCE_RUN_STATE = True` stale Colab visible-cell degerlerinin 48-row customer manifesti yeniden kosturmasini engeller.
- `M2_BATCH_SIZE = 4` official M2 kosusunda router adimini batch calistirir; OOM olursa runner batch'i dusurup retry yapar.
- Full controlled demo kosusu icin `M2_DEMO_LIMIT = None` varsayilan kalmalidir.
- `M2_ADAPTER_BATCH_SIZE = 2` ayni crop/part adapter tahminlerini dusuk batch ile calistirir; OOM olursa runner satir-bazli yola duser.
- `M2_HANDOFF_CACHE` degismeyen gorseller icin router/prototype handoff sonucunu yeniden kullanir; tam tazeleme icin `M2_REFRESH_HANDOFF_CACHE = True` yapin.
- Problem-only hizli tanilama icin `M2_RUN_PROBLEM_ONLY_DEMO = True` yapin; bu final kanit degildir, sadece 89 router/prototype zor ornegini kosar.
- Open-world sonuc klasoru `docs/demo_results/router_open_world/<timestamp>/` altina yazilir ve uygunsa push edilir.
- Tek gorsel denemesi yapmak isterseniz `RUN_SINGLE_IMAGE_DEMO = True` yapin ve `IMAGE_PATH` verin ya da upload kullanin.
- Router belirsiz kalirsa adapter zorlanmaz; sonuc `router_uncertain`, `unknown_crop` veya ilgili reddetme statusu olarak kalir.
- Adapter yolu varsayilan olarak `runs` kokunden cozulur; gerekiyorsa `ADAPTER_ROOT` verin.
- Prototype reconciler tek-gorsel ve M2 full demo icin varsayilan aciktir; M2 hedef-bazli policy varsa global policy yokken de acik kalir.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell01_bootstrap.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Notebook 1 ile ayni router kullanici ayarlari.
NOTEBOOK_PROFILE = 'a100_colab_default'  # custom, a100_colab_default, cpu_debug
NOTEBOOK_SPEED_MODE = 'fast'  # fast, balanced, quality

# Gerekirse son override katmani (opsiyonel). Final demo icin HF token erken dogrulansin.
INFERENCE_OVERRIDES = {'REQUIRE_HF_LOGIN': True}

# Notebook 8 tek gorsel adimi ayarlari. M2 icin varsayilan olarak kapali.
RUN_SINGLE_IMAGE_DEMO = True
ADAPTER_ROOT = None
RETURN_OOD = True
PRINT_JSON_RESULT = False

# Taxonomy/prototype reconciler. Varsayilan acik; sadece debug icin kapatin.
ENABLE_PROTOTYPE_RECONCILER = True
ROUTER_PROTOTYPE_BANK = None
TAXONOMY_REGISTRY = None
PROTOTYPE_MIN_SIMILARITY = None
PROTOTYPE_MIN_MARGIN = None
PROTOTYPE_MIN_NEGATIVE_GAP = None

# M2 full demo ayarlari. Tum hucreleri calistirinca bu manifest otomatik denenir.
M2_AUTO_APPLY_RUN_STATE = True
M2_FORCE_RUN_STATE = False
M2_RUN_STATE_CONFIG = 'docs/notebook8_m2_run_state.json'
M2_RUN_FULL_DEMO = False
M2_OPEN_WORLD_ONLY = False
M2_RUN_PROBLEM_ONLY_DEMO = False
M2_DEMO_MANIFEST = 'docs/demo_assets/m2_full_image_set/manifests/m2_balanced_80_run_manifest.csv'
M2_PROBLEM_ONLY_MANIFEST = 'docs/demo_assets/m2_problem_only_manifests/20260628T113313Z_router_failures.csv'
M2_PROBLEM_ONLY_CALIBRATION_MANIFEST = 'docs/demo_assets/m2_full_image_set/manifests/m2_full_image_set_run_manifest.csv'
M2_PROBLEM_ONLY_COMPARISON_BASELINE = ''
M2_DEMO_OUTPUT = '.runtime_tmp/m2_demo_checklist_run.json'
M2_DEMO_MARKDOWN_OUTPUT = '.runtime_tmp/m2_demo_checklist_run.md'
M2_DEMO_LIMIT = None
M2_BATCH_SIZE = 4
M2_ADAPTER_BATCH_SIZE = 2
M2_HANDOFF_CACHE = '.runtime_tmp/m2_balanced_80_handoff_cache.json'
M2_REFRESH_HANDOFF_CACHE = True
M2_STOP_ON_DEPENDENCY_BLOCKER = True
M2_AUTO_PUSH_RESULTS = False
M2_AUTO_PUSH_REMOTE_NAME = 'origin'
M2_AUTO_PUSH_BRANCH = 'master'
M2_REPO_RESULTS_ROOT = 'docs/demo_results/m2'
M2_COMPARISON_BASELINE = ''
M2_RUN_OPEN_WORLD_ROUTER_VALIDATION = False
M2_OPEN_WORLD_SUPPORTED_MANIFEST = 'docs/demo_assets/m2_full_image_set/manifests/m2_balanced_80_run_manifest.csv'
M2_OPEN_WORLD_MANIFEST = 'docs/demo_assets/open_world_router/manifests/m2_open_world_router_manifest.csv'
M2_OPEN_WORLD_OUTPUT_ROOT = 'docs/demo_results/router_open_world'
M2_OPEN_WORLD_HANDOFF_CACHE = '.runtime_tmp/router_open_world_handoff_cache.json'
M2_OPEN_WORLD_BASELINE_SUMMARY = M2_COMPARISON_BASELINE
M2_OPEN_WORLD_PROTOTYPE_ARTIFACT_DIR = 'docs/demo_results/m2/20260630T192242Z'
M2_OPEN_WORLD_MIN_ROWS = 300
M2_OPEN_WORLD_MIN_SUPPORTED_ROUTE_COVERAGE = 0.80
M2_OPEN_WORLD_REQUIRE_LATENCY_BASELINE = True
M2_OPEN_WORLD_FAIL_ON_NOT_READY = True
M2_AUTO_DISCONNECT_RUNTIME = False
M2_AUTO_DISCONNECT_GRACE_SECONDS = 20
M2_ENABLE_PROTOTYPE_RECONCILER = True
M2_AUTO_BUILD_PROTOTYPES = True
M2_PROTOTYPE_RUN_ID = ''
M2_PROTOTYPE_EMBEDDING_BACKEND = 'bioclip_open_clip'
M2_PROTOTYPE_EMBEDDING_MODEL_ID = 'imageomics/bioclip-2.5-vith14'
M2_PROTOTYPE_EMBEDDING_DEVICE = 'cuda'
M2_REUSE_EXISTING_PROTOTYPES = True
M2_REUSE_EXISTING_PROTOTYPE_CALIBRATION = True
M2_PROTOTYPE_MAX_IMAGES_PER_CLASS = 50
M2_PROTOTYPE_CURATION_ROOT = 'docs/demo_assets/prototype_curation/20260704T095107Z_router_refinement'
M2_PROTOTYPE_BANK = ''
M2_TAXONOMY_REGISTRY = ''
M2_PROTOTYPE_MIN_SIMILARITY = None
M2_PROTOTYPE_MIN_MARGIN = None
M2_PROTOTYPE_MIN_NEGATIVE_GAP = None
M2_AUTO_CALIBRATE_PROTOTYPE_RECONCILER = True
M2_AUTO_CALIBRATE_PROTOTYPES = M2_AUTO_CALIBRATE_PROTOTYPE_RECONCILER
M2_REQUIRE_CALIBRATED_PROTOTYPE_POLICY = True
M2_PROTOTYPE_CALIBRATION_OUTPUT = '.runtime_tmp/router_prototype_calibration.json'
M2_PROTOTYPE_CALIBRATION_LIMIT = None
M2_PROTOTYPE_CALIBRATION_MIN_PRECISION = 0.985
M2_PROTOTYPE_CALIBRATION_MIN_COVERAGE = 0.80
M2_PROTOTYPE_CALIBRATION_MAX_NEGATIVE_FALSE_ACCEPTS = 0
M2_PROTOTYPE_CALIBRATION_MAX_NEGATIVE_FALSE_ACCEPT_RATE = 0.05
M2_PROTOTYPE_TARGET_MIN_PRECISION = 0.98
M2_PROTOTYPE_TARGET_MAX_SUPPORTED_WRONG = 1
M2_PROTOTYPE_TARGET_MAX_CROSS_PART_SUPPORTED_WRONG = 0
M2_PROTOTYPE_TARGET_CLASS_MIN_ACCEPTED = 5
M2_PROTOTYPE_SIMILARITY_GRID = '0.20,0.30,0.40,0.50,0.60,0.70'
M2_PROTOTYPE_MARGIN_GRID = '0.00,0.02,0.04,0.06,0.08,0.10'
M2_PROTOTYPE_NEGATIVE_GAP_GRID = '0.00,0.02,0.04,0.06,0.08,0.10'
M2_ALLOW_NON_PLANT_FALSE_ACCEPTS = False
M2_PROTOTYPE_TARGET_POLICY_NEGATIVE_MODE = 'none'

run_cell_script('nb1_cell03_runtime_setup.py', globals())
run_cell_script('nb8_cell04_adapter_release_fetch.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Opsiyonel tek gorsel akisi. M2 full demo icin varsayilan olarak atlanir.
if RUN_SINGLE_IMAGE_DEMO:
    ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Yeni dosya yolu verin veya None birakip yukleme kullanin.

    run_cell_script('nb1_cell04_analysis.py', globals())

    router_result = result
    run_cell_script('nb8_cell05_adapter_prediction.py', globals())
else:
    ANALYSIS_IMAGE_PATH = None
    router_result = None
    auto_result = None
    print('[AUTO] Single-image demo skipped. Set RUN_SINGLE_IMAGE_DEMO=True to use this cell.')

auto_result


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

run_cell_script('nb8_cell06_m2_full_demo_run.py', globals())
